In [1]:
import pandas as pd
import numpy as np
import time

import sys
sys.path.insert(0, '../..')

from sn_clf.scripts.utils import get_sn_label_from_akb, load_features, download_akb_json, plot_config, get_art_label_from_akb, convert_to_onnx

In [2]:
#Crossmatch ojects from BTS and OIDs file
oids, features = load_features('../../dr23-features/sid_snad_clf_r_100.dat', '../../dr23-features/feature_snad_clf_r_100.dat')
bts_sn = pd.read_csv('../data/bts_crossmatched_2sec.csv')


#t = time.monotonic()
#bts_oids = list(bts_sn['OID'])
#crossmatch = np.isin(oids, bts_oids)
#print(f'Crossmatched in {np.round((time.monotonic() - t) / 60)} min')
#np.save(f'../data/bts_dr23_crossmatch.npy', crossmatch)

crossmatch = np.load(f'../data/bts_dr23_crossmatch.npy')

In [3]:
#Choose features, which will be used in training process
feature_names = '../../dr23-features/feature_snad_clf_r_100.name'
with open(feature_names) as f:
    names = f.read().split()

In [4]:
akb_oid, art_label = get_art_label_from_akb(f'../data/akb_objects.json')

akb_crossmatch = np.load(f'../data/akb_all_dr23_crossmatch.npy')    
akb_features = features[akb_crossmatch]



akb_features = pd.DataFrame(data=akb_features, columns=names)
akb_oids = oids[akb_crossmatch]
akb_art_lab = []
for oid in akb_oids:
    akb_art_lab.append(art_label[akb_oid == oid][0])
akb_art_lab = np.array(akb_art_lab)
akb_features['artefact'] = 1 - akb_art_lab
akb_features['oid'] = akb_oids

_, akb_sn_label = get_sn_label_from_akb('../data/akb_objects.json')
akb_sn_crossmatch = np.load(f'../data/akb_sn_dr23_crossmatch.npy')
akb_sn_oid = oids[akb_sn_crossmatch]
sn_mask = np.isin(akb_oids, akb_sn_oid)
is_sn = np.zeros(akb_oids.shape)
is_sn[sn_mask] = np.ones(is_sn[sn_mask].shape)

akb_features['is_sn'] = is_sn


akb_ot = np.zeros(len(akb_features))
akb_ot[(akb_features['artefact'] == 0) * (akb_features['is_sn'] == 0)] = np.ones(sum((akb_features['artefact'] == 0) * (akb_features['is_sn'] == 0)))
akb_features['akb_ot'] = akb_ot
akb_features['regular_obj'] = np.zeros(len(akb_features))

In [5]:
random_seed = 42

validated = pd.read_csv('bts_sample_oid.csv')
validated = validated[validated['is_sn'] == 1]['url'].apply(lambda url: url.split('/')[-1]).astype(np.uint64)

val_mask = np.isin(oids[crossmatch], validated)
bts_features = features[crossmatch][val_mask] # содержит только SN
bts_features = pd.DataFrame(data=bts_features, columns=names)
bts_features['oid'] = oids[crossmatch][val_mask]
bts_features['artefact'] = np.zeros(len(bts_features))
bts_features['is_sn'] = np.ones(len(bts_features))
bts_features['regular_obj'] = np.zeros(len(bts_features))
bts_features['akb_ot'] = np.zeros(len(bts_features))

# тут в качестве негативного класса используются рандомные объекты из дата релиза
indx = np.random.choice(np.arange(len(oids)), 10000)
regular_obj_feat = features[indx]
regular_obj_oid = oids[indx]
regular_obj = pd.DataFrame(data=regular_obj_feat, columns=names)
regular_obj['oid'] = regular_obj_oid
regular_obj['artefact'] = np.zeros(len(regular_obj))
regular_obj['is_sn'] = np.zeros(len(regular_obj))
regular_obj['akb_ot'] = np.zeros(len(regular_obj))
regular_obj['regular_obj'] = np.ones(len(regular_obj))

In [6]:
train_data = pd.concat([bts_features, 
                        akb_features[akb_features['is_sn'] == 0], # все из акб кроме сн
                        regular_obj])


print(f"Artefact fraction in train sample:{((train_data['artefact'] == 1).sum() / len(train_data)):.2f}")

print(f"SN fraction in train sample:{((train_data['is_sn'] == 1).sum() / len(train_data)):.2f}")
print(f"AKB other types fraction in train sample:{((train_data['akb_ot'] == 1).sum() / len(train_data)):.2f}")
print(f"regular object fraction in train sample:{((train_data['regular_obj'] == 1).sum() / len(train_data)):.2f}")

print(f'Train size: {len(train_data)}')

Artefact fraction in train sample:0.21
SN fraction in train sample:0.04
AKB other types fraction in train sample:0.16
regular object fraction in train sample:0.59
Train size: 16974


In [7]:
train_data['label'] = np.argmax(train_data[['artefact', 'is_sn', 'regular_obj', 'akb_ot']] .values, axis=1)

In [ ]:
#train_data.to_csv('../data/train_data_big.csv', index=False)